# M2.Ex4 — Automatic Speech Recognition (ASR)

## Use case: YouTube Lecture Study Assistant

Given a YouTube URL, this notebook builds a small pipeline that:

1. **Fast path** — try to fetch the official YouTube transcript (free, instant).
2. **Fallback path** — if the video has no transcript, download the audio with `yt-dlp` and transcribe it locally using **Docling's ASR pipeline** (Whisper Turbo under the hood).
3. **LLM step** — feed the transcript to an LLM to:
   - produce structured study notes (summary + key concepts), and
   - answer free-form questions about the video.

This is practical for a student who wants to study lectures, podcasts, or conference talks on YouTube — even ones with disabled captions.

## 1. Setup

Run the cell below once. `ffmpeg` must be installed on your system for Docling to handle audio formats other than WAV/MP3 — on macOS: `brew install ffmpeg`, on Debian/Ubuntu: `sudo apt-get install ffmpeg`.

In [ ]:
# Uncomment to install. The docling[asr] extra is the important one.
# %pip install -q youtube-transcript-api yt-dlp "docling[asr]" langchain-openai python-dotenv

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # expects OPENAI_API_KEY in a local .env file

WORK_DIR = Path("./asr_work")
WORK_DIR.mkdir(exist_ok=True)

## 2. Helper — extract the video ID from any YouTube URL

YouTube URLs come in many shapes (`youtu.be/...`, `youtube.com/watch?v=...`, `youtube.com/shorts/...`). We need a single ID to pass around.

In [ ]:
def extract_video_id(url: str) -> str:
    """Pull the 11-char video ID out of a YouTube URL."""
    patterns = [
        r"(?:v=|/v/|youtu\.be/|/embed/|/shorts/)([A-Za-z0-9_-]{11})",
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    raise ValueError(f"Could not extract a video ID from: {url}")


# Quick sanity check
assert extract_video_id("https://www.youtube.com/watch?v=dQw4w9WgXcQ") == "dQw4w9WgXcQ"
assert extract_video_id("https://youtu.be/dQw4w9WgXcQ") == "dQw4w9WgXcQ"
print("video ID extractor works ✓")

## 3. Fast path — fetch the official YouTube transcript

We use [`youtube-transcript-api`](https://pypi.org/project/youtube-transcript-api/), a tiny package that talks to YouTube's caption endpoint directly. No API key needed.

It raises an exception when the video has captions disabled or none uploaded — that's our cue to fall back to ASR.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound


def fetch_youtube_transcript(video_id: str, languages=("en",)) -> str | None:
    """
    Try to fetch the transcript via YouTube's own caption track.
    Returns the transcript text, or None if no captions exist.
    """
    try:
        api = YouTubeTranscriptApi()
        fetched = api.fetch(video_id, languages=list(languages))
        # fetched is iterable of snippet objects with .text and .start
        return " ".join(snippet.text for snippet in fetched)
    except (TranscriptsDisabled, NoTranscriptFound):
        return None
    except Exception as e:
        # Treat any other failure as "no transcript available" so we fall back.
        print(f"[transcript api] could not fetch captions: {e}")
        return None

## 4. Fallback path — download audio + transcribe with Docling

When the video has no captions we:

1. Use `yt-dlp` to download just the audio track as an MP3.
2. Hand the MP3 to Docling's `AsrPipeline`, which runs Whisper Turbo locally and returns a `DoclingDocument` with timestamped paragraphs.
3. Export the document to plain Markdown to use as our transcript.

Docling auto-selects `mlx-whisper` on Apple Silicon and falls back to native Whisper elsewhere — we don't have to configure anything.

In [ ]:
import yt_dlp


def download_audio(video_url: str, out_dir: Path) -> Path:
    """Download the audio track of a YouTube video as an MP3 file."""
    out_template = str(out_dir / "%(id)s.%(ext)s")
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": out_template,
        "quiet": True,
        "no_warnings": True,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=True)
    return out_dir / f"{info['id']}.mp3"

In [ ]:
from docling.datamodel import asr_model_specs
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import AsrPipelineOptions
from docling.document_converter import AudioFormatOption, DocumentConverter
from docling.pipeline.asr_pipeline import AsrPipeline


def transcribe_with_docling(audio_path: Path) -> str:
    """Run Docling's ASR pipeline on a local audio file and return Markdown text."""
    pipeline_options = AsrPipelineOptions()
    pipeline_options.asr_options = asr_model_specs.WHISPER_TURBO

    converter = DocumentConverter(
        format_options={
            InputFormat.AUDIO: AudioFormatOption(
                pipeline_cls=AsrPipeline,
                pipeline_options=pipeline_options,
            )
        }
    )
    result = converter.convert(audio_path)
    return result.document.export_to_markdown()

## 5. Unified function with automatic fallback

This is the public function for the rest of the notebook. It hides whether the transcript came from YouTube's captions or from Whisper.

In [ ]:
def get_transcript(video_url: str) -> tuple[str, str]:
    """
    Returns (transcript_text, source).
    `source` is either 'youtube_captions' or 'docling_asr'.
    """
    video_id = extract_video_id(video_url)

    # 1) Fast path
    captions = fetch_youtube_transcript(video_id)
    if captions:
        return captions, "youtube_captions"

    # 2) Fallback path
    print("No YouTube captions found — running Docling ASR (this may take a few minutes)...")
    audio_path = download_audio(video_url, WORK_DIR)
    transcript = transcribe_with_docling(audio_path)
    return transcript, "docling_asr"

## 6. LLM step — study notes & Q&A

Once we have the transcript text, the source no longer matters. We feed it to an LLM with two prompts: one for structured notes, one for free-form questions.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

NOTES_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a study assistant. Given a transcript of a video lecture or talk, "
     "produce concise study notes in Markdown with three sections: "
     "## Summary (3-5 sentences), ## Key Concepts (bullet list), "
     "## Questions to test understanding (3 questions). "
     "Be faithful to the transcript — do not invent facts."),
    ("human", "Transcript:\n\n{transcript}")
])

QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's question using only the transcript below. "
     "If the transcript does not contain the answer, say so honestly."),
    ("human", "Transcript:\n\n{transcript}\n\nQuestion: {question}")
])


def make_study_notes(transcript: str) -> str:
    msg = (NOTES_PROMPT | llm).invoke({"transcript": transcript})
    return msg.content


def ask(transcript: str, question: str) -> str:
    msg = (QA_PROMPT | llm).invoke({"transcript": transcript, "question": question})
    return msg.content

## 7. Demo A — video that has captions (fast path)

Most popular educational videos on YouTube have auto-generated or human-uploaded captions, so the fast path will be used.

In [ ]:
# A short, well-known talk that has captions
VIDEO_WITH_CAPTIONS = "https://www.youtube.com/watch?v=aircAruvnKk"  # 3Blue1Brown — 'But what is a neural network?'

transcript_a, source_a = get_transcript(VIDEO_WITH_CAPTIONS)
print(f"Source: {source_a}")
print(f"Transcript length: {len(transcript_a)} chars")
print("\nFirst 400 characters:\n")
print(transcript_a[:400])

In [ ]:
notes_a = make_study_notes(transcript_a)
print(notes_a)

In [ ]:
answer = ask(transcript_a, "What is a neuron in this context?")
print(answer)

## 8. Demo B — forcing the ASR fallback

To exercise the Docling path, we bypass the captions check and run ASR directly on a short video. This shows that even when YouTube captions are unavailable or disabled, the pipeline still produces a usable transcript.

> ⚠️ This step downloads the audio and runs Whisper locally — expect it to take a few minutes for a multi-minute video the first time (Whisper weights are downloaded on first run).

In [ ]:
# Pick a SHORT video for the demo so the ASR run finishes quickly.
SHORT_VIDEO = "https://www.youtube.com/watch?v=aircAruvnKk"  # replace with any short clip

audio_file = download_audio(SHORT_VIDEO, WORK_DIR)
print(f"Downloaded: {audio_file}")

transcript_b = transcribe_with_docling(audio_file)
print("\nFirst 500 characters of the ASR transcript:\n")
print(transcript_b[:500])

In [ ]:
notes_b = make_study_notes(transcript_b)
print(notes_b)

## 9. Wrap-up

What we built:

- A single `get_transcript(url)` that **always** returns text — falling back gracefully from YouTube captions to local Whisper ASR via Docling.
- An LLM layer on top that turns the transcript into study notes and answers questions.

Possible extensions for further practice:

- Cache transcripts on disk so a video is only transcribed once.
- Chunk long transcripts and build a small RAG index instead of stuffing the whole thing into the prompt.
- Add language detection + translation for non-English videos.
- Wrap the whole thing in a small Streamlit or Gradio UI.